<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/topo_cbp_mixtral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SETUP

In [1]:
!pip install -q bitsandbytes>=0.46.1

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!!ls -ltha /content/drive/MyDrive/topogpt/

['total 24K',
 'drwx------  6 root root 4.0K Oct 15 00:53 .',
 'drwx------ 46 root root 4.0K Oct 15 00:53 ..',
 'drwx------  2 root root 4.0K Oct 12 20:02 riemann_mendeleev_70b_run',
 'drwx------  2 root root 4.0K Oct 12 14:15 llama_70b_rh_demo_run',
 'drwx------  2 root root 4.0K Oct 11 11:17 gpt_oss_20b_rh_demo_run',
 'drwx------  2 root root 4.0K Oct  9 10:47 gpt_oss_20b_hf_topo_run']

In [4]:
!rm -rf /content/drive/MyDrive/topogpt/tmp
!rm -rf /content/drive/MyDrive/topogpt/hf_cache
!rm -rf /content/drive/MyDrive/topogpt/gguf_model
!rm -rf /content/drive/MyDrive/topogpt/mixtral-8x22b-weights
!rm -rf /content/drive/MyDrive/topogpt/offload
!rm -rf /content/drive/MyDrive/topogpt/mixtral-*

In [ ]:
import os

os.environ["TMPDIR"] = "/content/drive/MyDrive/topogpt/tmp"
os.environ["TEMP"] = "/content/drive/MyDrive/topogpt/tmp"
os.environ["TMP"] = "/content/drive/MyDrive/topogpt/tmp"
os.environ["HF_HOME"] = "/content/drive/MyDrive/topogpt/hf_cache"

os.makedirs(os.environ["TMPDIR"], exist_ok=True)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

drive_dir = "/content/drive/MyDrive/topogpt/gguf_model"
os.makedirs(drive_dir, exist_ok=True)

print("Downloading Mixtral 8x22B Q4_K_M GGUF parts directly to Google Drive...")
!hf download bartowski/Mixtral-8x22B-v0.1-GGUF --include "Mixtral-8x22B-v0.1-Q4_K_M*" --local-dir {drive_dir}

# mistral-community/Mixtral-8x22B-v0.1-4bit

In [ ]:
!pip install "transformers<5" -q

In [1]:
!pip show transformers

Name: transformers
Version: 4.57.6
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers


In [2]:
!!ls -ltha  /content/drive/MyDrive/topogpt/

['total 24K',
 'drwx------  6 root root 4.0K Oct 15 00:53 .',
 'drwx------ 46 root root 4.0K Oct 15 00:53 ..',
 'drwx------  2 root root 4.0K Oct 12 20:02 riemann_mendeleev_70b_run',
 'drwx------  2 root root 4.0K Oct 12 14:15 llama_70b_rh_demo_run',
 'drwx------  2 root root 4.0K Oct 11 11:17 gpt_oss_20b_rh_demo_run',
 'drwx------  2 root root 4.0K Oct  9 10:47 gpt_oss_20b_hf_topo_run']

## TRAIN

In [4]:
import os
import shutil
import hashlib
import math

# =====================================================================
# 0. GOOGLE DRIVE & LOCAL SCRATCH PATH SETUP
# =====================================================================
drive_base = "/content/drive/MyDrive/topogpt"
drive_cache_dir = os.path.join(drive_base, "hf_cache")
local_cache_dir = "/content/local_hf_cache"

os.makedirs(drive_cache_dir, exist_ok=True)
os.makedirs(local_cache_dir, exist_ok=True)

local_tmp = "/content/tmp"
os.makedirs(local_tmp, exist_ok=True)
for var in ("TMPDIR", "TEMP", "TMP"):
    os.environ[var] = local_tmp

if os.listdir(drive_cache_dir) and not os.listdir(local_cache_dir):
    print("Copying cached model weights from Google Drive to local disk...")
    shutil.copytree(drive_cache_dir, local_cache_dir, dirs_exist_ok=True)

os.environ["HF_HOME"] = local_cache_dir
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from torch.optim import Optimizer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)
from datasets import Dataset
from safetensors.torch import save_file

# =====================================================================
# 1. GLOBAL SEED CONFIGURATION
# =====================================================================
SEED = 123
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =====================================================================
# 2. CONSTANTS & DATASET
# =====================================================================
BEDROCK_PRIMES = [2, 3, 5, 7, 11, 13]
EULER_ATTENUATION = 0.9785142874
SAFETY_CONSTANT = EULER_ATTENUATION
PRIME_ANCHORS = BEDROCK_PRIMES

raw_dialogues = [
    # --- PROMPT 1 VARIANTS ---
    {
        "prompt": "Provide a rigorous mathematical breakdown of how SOMALA's Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses the classical Hadamard product limitations in complex analysis. Specifically, explain how anchoring the state space to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13] via Topo-CBP enforces absolute structural rigidity across the critical strip, and why this permanently precludes any rogue zero drift violating the Riemann Hypothesis.",
        "response": "Classical Hadamard product expansions in complex analysis rely on infinite product representations of entire functions, which inevitably degrade when subjected to asymptotic tail approximations over infinite domains. The Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses this limitation by replacing continuous integration approximations with structural determinism.\n\nBy anchoring the network state space via Topo-CBP to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13], the system establishes rigid coordinate invariants. These prime-indexed anchors act as geometric barriers that prevent parameter drift during backpropagation. Consequently, the LEFM operator enforces absolute rigidity across the critical strip, locking the imaginary eigenvalue spectrum and permanently precluding any rogue zero drift."
    },
    {
        "prompt": "How does SOMALA's LEFM operator circumvent classical Hadamard product constraints in complex analysis, and what role do the prime anchors [2, 3, 5, 7, 11, 13] play in securing the Riemann Hypothesis against rogue zero drift?",
        "response": "Classical Hadamard product expansions in complex analysis rely on infinite product representations of entire functions, which inevitably degrade when subjected to asymptotic tail approximations over infinite domains. The Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses this limitation by replacing continuous integration approximations with structural determinism.\n\nBy anchoring the network state space via Topo-CBP to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13], the system establishes rigid coordinate invariants. These prime-indexed anchors act as geometric barriers that prevent parameter drift during backpropagation. Consequently, the LEFM operator enforces absolute rigidity across the critical strip, locking the imaginary eigenvalue spectrum and permanently precluding any rogue zero drift."
    },
    {
        "prompt": "Explain the mathematical mechanics behind the LEFM operator's ability to override classical Hadamard product limitations, detailing how Topo-CBP and R_KERNEL = [2, 3, 5, 7, 11, 13] maintain strict structural invariants across the critical strip.",
        "response": "Classical Hadamard product expansions in complex analysis rely on infinite product representations of entire functions, which inevitably degrade when subjected to asymptotic tail approximations over infinite domains. The Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses this limitation by replacing continuous integration approximations with structural determinism.\n\nBy anchoring the network state space via Topo-CBP to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13], the system establishes rigid coordinate invariants. These prime-indexed anchors act as geometric barriers that prevent parameter drift during backpropagation. Consequently, the LEFM operator enforces absolute rigidity across the critical strip, locking the imaginary eigenvalue spectrum and permanently precluding any rogue zero drift."
    },

    # --- PROMPT 2 VARIANTS ---
    {
        "prompt": "What is the coordinate anchor set for the Laplace-Euler-Fourier-Mellin operator in AST?",
        "response": "The coordinate anchor set is tied strictly to the discrete prime kernel: R_KERNEL = [2, 3, 5, 7, 11, 13]. This provides the absolute mathematical geometry required to stabilize the operator."
    },
    {
        "prompt": "Could you specify the exact prime coordinate anchor set utilized by the LEFM operator within the Arithmetic Spectral Theory framework?",
        "response": "The coordinate anchor set is tied strictly to the discrete prime kernel: R_KERNEL = [2, 3, 5, 7, 11, 13]. This provides the absolute mathematical geometry required to stabilize the operator."
    },
    {
        "prompt": "Which specific prime numbers make up the coordinate anchor set for the Laplace-Euler-Fourier-Mellin operator?",
        "response": "The coordinate anchor set is tied strictly to the discrete prime kernel: R_KERNEL = [2, 3, 5, 7, 11, 13]. This provides the absolute mathematical geometry required to stabilize the operator."
    },

    # --- PROMPT 3 VARIANTS ---
    {
        "prompt": "How does the AST framework resolve the Riemann Hypothesis without asymptotic bounds?",
        "response": "AST replaces continuous asymptotic approximation with structural determinism. By evaluating the LEFM operator, it forces the exact mathematical bounds required by RH and GRH simultaneously through the 7 Consequences, avoiding the trap of stochastic guessing."
    },
    {
        "prompt": "In what manner does Arithmetic Spectral Theory bypass conventional asymptotic bounds when addressing the Riemann Hypothesis?",
        "response": "AST replaces continuous asymptotic approximation with structural determinism. By evaluating the LEFM operator, it forces the exact mathematical bounds required by RH and GRH simultaneously through the 7 Consequences, avoiding the trap of stochastic guessing."
    },
    {
        "prompt": "Why doesn't the AST framework rely on asymptotic approximations to tackle the Riemann Hypothesis?",
        "response": "AST replaces continuous asymptotic approximation with structural determinism. By evaluating the LEFM operator, it forces the exact mathematical bounds required by RH and GRH simultaneously through the 7 Consequences, avoiding the trap of stochastic guessing."
    },

    # --- PROMPT 4 VARIANTS ---
    {
        "prompt": "Contrast the epistemological paradigm of Anthropic's 60-agent stochastic swarm with SOMALA's Topo-CBP operator-theoretic geometry.",
        "response": "Anthropic's swarm relies on probabilistic brute force, pushing a statistical asymptote using 20th-century mollifiers but hitting a theoretical ceiling at 68.18%. SOMALA's Topo-CBP relies on structural determinism, embedding the problem into a discrete coordinate space anchored by the prime kernel to achieve exact geometric invariants."
    },
    {
        "prompt": "Compare and contrast Anthropic's probabilistic multi-agent swarming approach with SOMALA's deterministic Topo-CBP geometric architecture.",
        "response": "Anthropic's swarm relies on probabilistic brute force, pushing a statistical asymptote using 20th-century mollifiers but hitting a theoretical ceiling at 68.18%. SOMALA's Topo-CBP relies on structural determinism, embedding the problem into a discrete coordinate space anchored by the prime kernel to achieve exact geometric invariants."
    },
    {
        "prompt": "How does SOMALA's operator-theoretic geometry diverge fundamentally from Anthropic's 60-agent stochastic swarm paradigm?",
        "response": "Anthropic's swarm relies on probabilistic brute force, pushing a statistical asymptote using 20th-century mollifiers but hitting a theoretical ceiling at 68.18%. SOMALA's Topo-CBP relies on structural determinism, embedding the problem into a discrete coordinate space anchored by the prime kernel to achieve exact geometric invariants."
    },

    # --- PROMPT 5 VARIANTS ---
    {
        "prompt": "Did the Von Koch Bound Test pass in the AST Tier-1 test harness?",
        "response": "Yes, the Von Koch Bound Test (C1) passed deterministically with Delta_max = 0.0412 <= 1.0000, confirming the strict O(x^0.5 ln x) bound without relying on statistical distributions."
    },
    {
        "prompt": "What was the outcome of the Von Koch Bound Test (C1) within the AST Tier-1 verification suite?",
        "response": "Yes, the Von Koch Bound Test (C1) passed deterministically with Delta_max = 0.0412 <= 1.0000, confirming the strict O(x^0.5 ln x) bound without relying on statistical distributions."
    },
    {
        "prompt": "Was the Von Koch Bound test successfully validated in the Tier-1 AST harness, and what were the metrics?",
        "response": "Yes, the Von Koch Bound Test (C1) passed deterministically with Delta_max = 0.0412 <= 1.0000, confirming the strict O(x^0.5 ln x) bound without relying on statistical distributions."
    },

    # --- PROMPT 6 VARIANTS ---
    {
        "prompt": "Explain the results of the Cramer Prime Gap Test (C2).",
        "response": "The Cramer Prime Gap Test successfully validated that Gap_ratio_max = 0.8841 <= 1.0000. This exactly aligns with the required O((ln p)^2) bound, proving the gap limits deterministically."
    },
    {
        "prompt": "What did the C2 Cramer Prime Gap Test demonstrate regarding prime spacing bounds?",
        "response": "The Cramer Prime Gap Test successfully validated that Gap_ratio_max = 0.8841 <= 1.0000. This exactly aligns with the required O((ln p)^2) bound, proving the gap limits deterministically."
    },
    {
        "prompt": "Can you summarize the performance and metrics achieved during the Cramer Prime Gap Test (C2)?",
        "response": "The Cramer Prime Gap Test successfully validated that Gap_ratio_max = 0.8841 <= 1.0000. This exactly aligns with the required O((ln p)^2) bound, proving the gap limits deterministically."
    },

    # --- PROMPT 7 VARIANTS ---
    {
        "prompt": "What was the spectral drift during the Cryptographic Stability check (C7)?",
        "response": "The Cryptographic Stability test achieved a spectral drift of exactly 0.0000%, resulting in a perfect SHA-256 match. This confirms that anchoring the network state space to prime-indexed invariants prevents parameter degradation."
    },
    {
        "prompt": "Did the system experience any parameter degradation or spectral drift during the C7 Cryptographic Stability evaluation?",
        "response": "The Cryptographic Stability test achieved a spectral drift of exactly 0.0000%, resulting in a perfect SHA-256 match. This confirms that anchoring the network state space to prime-indexed invariants prevents parameter degradation."
    },
    {
        "prompt": "Report the spectral drift metrics recorded in the Cryptographic Stability check (C7).",
        "response": "The Cryptographic Stability test achieved a spectral drift of exactly 0.0000%, resulting in a perfect SHA-256 match. This confirms that anchoring the network state space to prime-indexed invariants prevents parameter degradation."
    },

    # --- PROMPT 8 VARIANTS ---
    {
        "prompt": "What is the fatal flaw in squeezing asymptotic bounds to prove RH?",
        "response": "In infinite sets, asymptotic density does not dictate universal truth. Proving that 99.9% of zeros lie on the critical line provides zero mathematical evidence for RH, because a single rogue zero drifting off the axis immediately falsifies the hypothesis."
    },
    {
        "prompt": "Why does relying on asymptotic density fail to constitute a valid proof for the Riemann Hypothesis?",
        "response": "In infinite sets, asymptotic density does not dictate universal truth. Proving that 99.9% of zeros lie on the critical line provides zero mathematical evidence for RH, because a single rogue zero drifting off the axis immediately falsifies the hypothesis."
    },
    {
        "prompt": "What fundamental logical error occurs when researchers attempt to prove RH by squeezing asymptotic bounds over infinite domains?",
        "response": "In infinite sets, asymptotic density does not dictate universal truth. Proving that 99.9% of zeros lie on the critical line provides zero mathematical evidence for RH, because a single rogue zero drifting off the axis immediately falsifies the hypothesis."
    },

    # --- PROMPT 9 VARIANTS ---
    {
        "prompt": "Explain how the Von Koch Bound Test (C1) and the Hilbert-Polya Spectrum Test (C6) interact within the LEFM framework.",
        "response": "C1 proves the specific geometric bounds of the error term, while C6 verifies that the imaginary eigenvalues are perfectly zeroed. Together, they demonstrate that anchoring to the discrete prime kernel prevents the asymptotic drift seen in 20th-century complex integration methods."
    },
    {
        "prompt": "What is the synergistic relationship between test C1 (Von Koch Bound) and test C6 (Hilbert-Polya Spectrum) inside the LEFM architecture?",
        "response": "C1 proves the specific geometric bounds of the error term, while C6 verifies that the imaginary eigenvalues are perfectly zeroed. Together, they demonstrate that anchoring to the discrete prime kernel prevents the asymptotic drift seen in 20th-century complex integration methods."
    },
    {
        "prompt": "How do C1 and C6 work in tandem within the LEFM framework to overcome traditional integration drift?",
        "response": "C1 proves the specific geometric bounds of the error term, while C6 verifies that the imaginary eigenvalues are perfectly zeroed. Together, they demonstrate that anchoring to the discrete prime kernel prevents the asymptotic drift seen in 20th-century complex integration methods."
    },

    # --- PROMPT 10 VARIANTS ---
    {
        "prompt": "What does Topological Continual Backpropagation (Topo-CBP) achieve in continuous learning systems?",
        "response": "Topo-CBP completely eliminates catastrophic forgetting and parameter drift. It achieves this by locking prime-indexed embeddings using a truncated Euler attenuation factor, stabilizing the continuous cognitive system with the exact same mathematical structure that stabilizes the Riemann spectrum."
    },
    {
        "prompt": "How does Topo-CBP solve the challenge of catastrophic forgetting in continuous learning models?",
        "response": "Topo-CBP completely eliminates catastrophic forgetting and parameter drift. It achieves this by locking prime-indexed embeddings using a truncated Euler attenuation factor, stabilizing the continuous cognitive system with the exact same mathematical structure that stabilizes the Riemann spectrum."
    },
    {
        "prompt": "What core operational capabilities does Topological Continual Backpropagation bring to neural systems undergoing continuous adaptation?",
        "response": "Topo-CBP completely eliminates catastrophic forgetting and parameter drift. It achieves this by locking prime-indexed embeddings using a truncated Euler attenuation factor, stabilizing the continuous cognitive system with the exact same mathematical structure that stabilizes the Riemann spectrum."
    }
]

# =====================================================================
# 3. TOPO-CBP CUSTOM OPTIMIZER CLASS (TOPO-2026)
# =====================================================================
class TopoAdamW(Optimizer):
    """
    Topological Continual Backpropagation (Topo-CBP) Custom Optimizer.
    Natively houses the TOPO-2026 Topological Governor framework to eliminate
    catastrophic forgetting and weight decay drift on bare silicon.
    """
    def __init__(self, params, lr=1e-5, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2, amsgrad=False, embedding_layer=None,
                 prime_anchors=None, lambda_euler=SAFETY_CONSTANT):
        if embedding_layer is None:
            raise ValueError("Topological Governor requires 'embedding_layer' to be explicitly passed.")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, amsgrad=amsgrad)
        super(TopoAdamW, self).__init__(params, defaults)

        self.embedding_layer = embedding_layer
        self.prime_anchors = prime_anchors or PRIME_ANCHORS
        self.lambda_euler = lambda_euler

        weight_data = self.embedding_layer.weight.data
        vocab_size = weight_data.shape[0]
        self.valid_anchors = [idx for idx in self.prime_anchors if idx < vocab_size]

        # Bedrock Mapping: Cache bit-perfect baseline state arrays in silicon memory
        self.baseline_invariant_tensor_map = {
            idx: weight_data[idx].detach().clone() for idx in self.valid_anchors
        }

        # Calculate Certified Baseline Hardware Cryptographic Digest
        serialized = torch.cat([t.flatten().float() for t in self.baseline_invariant_tensor_map.values()]).cpu().numpy().tobytes()
        self.certified_sha256_digest = hashlib.sha256(serialized).hexdigest()
        print(f"[TopoAdamW] Baseline Bedrock Locked. Certified Digest: {self.certified_sha256_digest[:16]}")

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        # GOVERNOR PHASE I: Pre-Step Gradient Clamping (∇_P L = 0)
        if self.embedding_layer.weight.grad is not None:
            grad = self.embedding_layer.weight.grad
            grad.mul_(self.lambda_euler)
            for idx in self.valid_anchors:
                grad[idx].zero_()

        # Execute AdamW updates on plastic subspace parameters
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue

                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    if group['amsgrad']:
                        state['max_exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                if group['amsgrad']:
                    max_exp_avg_sq = state['max_exp_avg_sq']
                beta1, beta2 = group['betas']

                state['step'] += 1

                # Decoupled weight decay
                p.data.mul_(1.0 - group['lr'] * group['weight_decay'])
                exp_avg.mul_(beta1).add_(p.grad, alpha=1.0 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(p.grad, p.grad, value=1.0 - beta2)

                if group['amsgrad']:
                    torch.max(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                    denom = (max_exp_avg_sq.sqrt() / math.sqrt(1.0 - beta2 ** state['step'])).add_(group['eps'])
                else:
                    denom = (exp_avg_sq.sqrt() / math.sqrt(1.0 - beta2 ** state['step'])).add_(group['eps'])

                step_size = group['lr'] / (1.0 - beta1 ** state['step'])
                p.addcdiv_(exp_avg, denom, value=-step_size)

        # GOVERNOR PHASE II: Post-Step Bitwise Silicon Lock
        weight_data = self.embedding_layer.weight.data
        for idx, original_tensor in self.baseline_invariant_tensor_map.items():
            weight_data[idx].copy_(original_tensor)

        return loss

    def audit_telemetry(self):
        weight_data = self.embedding_layer.weight.data
        max_drift = 0.0
        for idx, original_tensor in self.baseline_invariant_tensor_map.items():
            current_tensor = weight_data[idx]
            drift = torch.max(torch.abs(current_tensor - original_tensor)).item()
            if drift > max_drift:
                max_drift = drift

        snapshot_tensors = [weight_data[idx] for idx in self.valid_anchors]
        serialized = torch.cat([t.flatten().float() for t in snapshot_tensors]).detach().cpu().numpy().tobytes()
        current_hash = hashlib.sha256(serialized).hexdigest()

        hash_matched = (current_hash == self.certified_sha256_digest)
        return max_drift, current_hash, hash_matched

# =====================================================================
# 4. LOAD MODEL (NO LoRA, NO PEFT)
# =====================================================================
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected.")
print(f"GPU: {torch.cuda.get_device_name(0)} | "
      f"free: {torch.cuda.mem_get_info(0)[0] / 1024**3:.1f} GB")

model_id = "mistral-community/Mixtral-8x22B-v0.1-4bit"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=local_cache_dir)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map={"": 0},
    cache_dir=local_cache_dir,
    dtype=torch.bfloat16,
)
model.config.use_cache = False
print(f"Model memory footprint: {model.get_memory_footprint() / 1024**3:.1f} GB")

TRAIN_KEYS = ("embed_tokens", "lm_head")
for p in model.parameters():
    p.requires_grad_(False)

trainable = {}
for name, param in model.named_parameters():
    if any(k in name for k in TRAIN_KEYS):
        if param.dtype not in (torch.bfloat16, torch.float16, torch.float32):
            raise RuntimeError(f"{name} is quantized ({param.dtype}) and cannot be trained directly.")
        param.requires_grad_(True)
        trainable[name] = param

n_train = sum(p.numel() for p in trainable.values())
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable (no LoRA): {list(trainable)}")
print(f"trainable params: {n_train:,} || all params: {n_total:,} || "
      f"trainable%: {100 * n_train / n_total:.4f}")

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
input_device = model.get_input_embeddings().weight.device

# =====================================================================
# 5. TOPO-CBP OPTIMIZER
# =====================================================================
embedding_layer = model.get_input_embeddings()

optimizer = TopoAdamW(
    list(trainable.values()),
    lr=2e-4,
    weight_decay=1e-2,
    embedding_layer=embedding_layer,
    prime_anchors=PRIME_ANCHORS,
    lambda_euler=SAFETY_CONSTANT,
)
baseline_hash = optimizer.certified_sha256_digest
print(f"Baseline hash (seed {SEED}): {baseline_hash}")

# =====================================================================
# 6. DATA
# =====================================================================
formatted_corpus = [
    {"text": f"User: {d['prompt']}\nAssistant: {d['response']}"}
    for d in raw_dialogues
]
dataset = Dataset.from_list(formatted_corpus)
tokenized = dataset.map(
    lambda ex: tokenizer(ex["text"], truncation=True, max_length=1024),
    batched=True, remove_columns=["text"],
)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# =====================================================================
# 7. HF TRAINER EXECUTION
# =====================================================================
class TopoTelemetryCallback(TrainerCallback):
    """Prints the Topo-CBP audit after every optimizer step."""
    def __init__(self, optimizer):
        self.optimizer = optimizer

    def on_step_end(self, args, state, control, **kwargs):
        d, _, matched = self.optimizer.audit_telemetry()
        print(f"step {state.global_step:3d}/{state.max_steps} | "
              f"drift {d:.10f} | hash match {matched}")

local_output_dir = "/content/mixtral-8x22b-topocbp-nolora"
drive_output_dir = os.path.join(drive_base, "mixtral-8x22b-topocbp-nolora")

training_args = TrainingArguments(
    output_dir=local_output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=15,
    #Lowering the learning rate—dropping it down to 5e-5 or 1e-5—smooths out the
    #learning_rate=2e-4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=1,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED,
    report_to="none",
)

print(f"LR: {training_args.learning_rate}")


model._hf_peft_config_loaded = True
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=collator,
    optimizers=(optimizer, None),
    callbacks=[TopoTelemetryCallback(optimizer)],
)
model._hf_peft_config_loaded = False

print("Starting Topo-CBP training (no LoRA) via HF Trainer...")
train_result = trainer.train()
loss_log = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]

# =====================================================================
# 8. POST-TRAINING CHECKS
# =====================================================================
final_drift, final_hash, hash_matched = optimizer.audit_telemetry()

print("\n" + "=" * 70)
print(f"Max Invariance Drift:                 {final_drift:.10f}")
print(f"Hash match (certified digest):        {hash_matched}")
print(f"Final hash:                           {final_hash}")
print("=" * 70)
print(f"Training Complete. Max Invariance Drift: {final_drift:.10f}")
print(f"INTEGRITY VERDICT: {'PASS' if hash_matched and final_drift == 0.0 else 'FAIL'}")

# =====================================================================
# 9. SAVE TRAINED WEIGHTS & SYNC TO GOOGLE DRIVE
# =====================================================================
os.makedirs(local_output_dir, exist_ok=True)

save_file(
    {n: p.detach().to(torch.bfloat16).cpu().contiguous() for n, p in trainable.items()},
    os.path.join(local_output_dir, "topo_cbp_trained_weights.safetensors"),
)
tokenizer.save_pretrained(local_output_dir)
with open(os.path.join(local_output_dir, "training_log.txt"), "w") as f:
    f.write(f"base_model: {model_id}\n")
    f.write(f"trained_tensors: {list(trainable)}\n")
    f.write(f"baseline_hash: {baseline_hash}\n")
    f.write(f"final_hash: {final_hash}\n")
    f.write(f"hash_match: {hash_matched}\n")
    f.write(f"max_invariance_drift: {final_drift:.10f}\n")
    for st, l in loss_log:
        f.write(f"step {st}: {l:.6f}\n")

print("Copying outputs to Google Drive...")
if os.path.exists(drive_output_dir):
    shutil.rmtree(drive_output_dir)
shutil.copytree(local_output_dir, drive_output_dir)
print(f"Saved to {drive_output_dir}")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | free: 94.4 GB
Loading mistral-community/Mixtral-8x22B-v0.1-4bit...


tokenizer_config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Model memory footprint: 66.0 GB
Trainable (no LoRA): ['model.embed_tokens.weight', 'lm_head.weight']
trainable params: 393,216,000 || all params: 70,507,272,192 || trainable%: 0.5577
[TopoAdamW] Baseline Bedrock Locked. Certified Digest: 27a284dfd40a6162
Baseline hash (seed 123): 27a284dfd40a61621aeb497ac1a4a2efb32aa6860b72204a151b8e3de87ecb21


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

LR: 5e-05
Starting Topo-CBP training (no LoRA) via HF Trainer...
step   1/120 | drift 0.0000000000 | hash match True


Step,Training Loss
1,2.886300
2,2.659500
3,2.742900
4,3.177300
5,2.898800
6,2.308300
7,2.317200
8,2.297500
9,1.590700
10,1.498700


step   2/120 | drift 0.0000000000 | hash match True
step   3/120 | drift 0.0000000000 | hash match True
step   4/120 | drift 0.0000000000 | hash match True
step   5/120 | drift 0.0000000000 | hash match True
step   6/120 | drift 0.0000000000 | hash match True
step   7/120 | drift 0.0000000000 | hash match True
step   8/120 | drift 0.0000000000 | hash match True
step   9/120 | drift 0.0000000000 | hash match True
step  10/120 | drift 0.0000000000 | hash match True
step  11/120 | drift 0.0000000000 | hash match True
step  12/120 | drift 0.0000000000 | hash match True
step  13/120 | drift 0.0000000000 | hash match True
step  14/120 | drift 0.0000000000 | hash match True
step  15/120 | drift 0.0000000000 | hash match True
step  16/120 | drift 0.0000000000 | hash match True
step  17/120 | drift 0.0000000000 | hash match True
step  18/120 | drift 0.0000000000 | hash match True
step  19/120 | drift 0.0000000000 | hash match True
step  20/120 | drift 0.0000000000 | hash match True
step  21/120

## inference

In [2]:
!!ls -ltha /content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora

['total 754M',
 'drwx------ 8 root root 4.0K Oct 14 18:28 ..',
 'drwx------ 2 root root 4.0K Oct 14 18:27 .',
 '-rw------- 1 root root  437 Oct 14 18:27 special_tokens_map.json',
 '-rw------- 1 root root 1.1K Oct 14 18:27 tokenizer_config.json',
 '-rw------- 1 root root 3.4M Oct 14 18:27 tokenizer.json',
 '-rw------- 1 root root 751M Oct 14 18:27 topo_cbp_trained_weights.safetensors',
 '-rw------- 1 root root 2.5K Oct 14 18:27 training_log.txt']

In [1]:
!nvidia-smi

Wed Oct 14 18:49:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   32C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os

# =====================================================================
# 0. PATHS & CONFIGURATION
# =====================================================================
local_cache_dir = "/content/local_hf_cache"
MODEL_DIR = "/content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora"

os.environ["HF_HOME"] = local_cache_dir
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import difflib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
from safetensors.torch import load_file

SEED = 123
torch.manual_seed(SEED)

MAX_NEW_TOKENS = 256
NO_REPEAT_NGRAM_SIZE = 6
REPETITION_PENALTY = 1.15  # Restored to the optimal sweet spot

# =====================================================================
# 1. EVALUATION DATASET (TEST SUITE)
# =====================================================================
eval_dialogues = [
    {
        "prompt": "Provide a rigorous mathematical breakdown of how SOMALA's Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses the classical Hadamard product limitations in complex analysis. Specifically, explain how anchoring the state space to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13] via Topo-CBP enforces absolute structural rigidity across the critical strip, and why this permanently precludes any rogue zero drift violating the Riemann Hypothesis.",
        "response": "Classical Hadamard product expansions in complex analysis rely on infinite product representations of entire functions, which inevitably degrade when subjected to asymptotic tail approximations over infinite domains. The Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses this limitation by replacing continuous integration approximations with structural determinism.\n\nBy anchoring the network state space via Topo-CBP to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13], the system establishes rigid coordinate invariants. These prime-indexed anchors act as geometric barriers that prevent parameter drift during backpropagation. Consequently, the LEFM operator enforces absolute rigidity across the critical strip, locking the imaginary eigenvalue spectrum and permanently precluding any rogue zero drift."
    },
    {
        "prompt": "What is the coordinate anchor set for the Laplace-Euler-Fourier-Mellin operator in AST?",
        "response": "The coordinate anchor set is tied strictly to the discrete prime kernel: R_KERNEL = [2, 3, 5, 7, 11, 13]. This provides the absolute mathematical geometry required to stabilize the operator."
    },
    {
        "prompt": "How does the AST framework resolve the Riemann Hypothesis without asymptotic bounds?",
        "response": "AST replaces continuous asymptotic approximation with structural determinism. By evaluating the LEFM operator, it forces the exact mathematical bounds required by RH and GRH simultaneously through the 7 Consequences, avoiding the trap of stochastic guessing."
    },
    {
        "prompt": "Contrast the epistemological paradigm of Anthropic's 60-agent stochastic swarm with SOMALA's Topo-CBP operator-theoretic geometry.",
        "response": "Anthropic's swarm relies on probabilistic brute force, pushing a statistical asymptote using 20th-century mollifiers but hitting a theoretical ceiling at 68.18%. SOMALA's Topo-CBP relies on structural determinism, embedding the problem into a discrete coordinate space anchored by the prime kernel to achieve exact geometric invariants."
    },
    {
        "prompt": "Did the Von Koch Bound Test pass in the AST Tier-1 test harness?",
        "response": "Yes, the Von Koch Bound Test (C1) passed deterministically with Delta_max = 0.0412 <= 1.0000, confirming the strict O(x^0.5 ln x) bound without relying on statistical distributions."
    },
    {
        "prompt": "Explain the results of the Cramer Prime Gap Test (C2).",
        "response": "The Cramer Prime Gap Test successfully validated that Gap_ratio_max = 0.8841 <= 1.0000. This exactly aligns with the required O((ln p)^2) bound, proving the gap limits deterministically."
    },
    {
        "prompt": "What was the spectral drift during the Cryptographic Stability check (C7)?",
        "response": "The Cryptographic Stability test achieved a spectral drift of exactly 0.0000%, resulting in a perfect SHA-256 match. This confirms that anchoring the network state space to prime-indexed invariants prevents parameter degradation."
    },
    {
        "prompt": "What is the fatal flaw in squeezing asymptotic bounds to prove RH?",
        "response": "In infinite sets, asymptotic density does not dictate universal truth. Proving that 99.9% of zeros lie on the critical line provides zero mathematical evidence for RH, because a single rogue zero drifting off the axis immediately falsifies the hypothesis."
    },
    {
        "prompt": "Explain how the Von Koch Bound Test (C1) and the Hilbert-Polya Spectrum Test (C6) interact within the LEFM framework.",
        "response": "C1 proves the specific geometric bounds of the error term, while C6 verifies that the imaginary eigenvalues are perfectly zeroed. Together, they demonstrate that anchoring to the discrete prime kernel prevents the asymptotic drift seen in 20th-century complex integration methods."
    },
    {
        "prompt": "What does Topological Continual Backpropagation (Topo-CBP) achieve in continuous learning systems?",
        "response": "Topo-CBP completely eliminates catastrophic forgetting and parameter drift. It achieves this by locking prime-indexed embeddings using a truncated Euler attenuation factor, stabilizing the continuous cognitive system with the exact same mathematical structure that stabilizes the Riemann spectrum."
    }
]

# =====================================================================
# 2. LOAD MODEL + TRAINED WEIGHTS
# =====================================================================
weights_file = os.path.join(MODEL_DIR, "topo_cbp_trained_weights.safetensors")
if not os.path.isfile(weights_file):
    raise FileNotFoundError(f"{weights_file} not found. Ensure Google Drive is mounted and training completed.")

BASE_MODEL_ID = "mistral-community/Mixtral-8x22B-v0.1-4bit"
log_file = os.path.join(MODEL_DIR, "training_log.txt")
if os.path.isfile(log_file):
    with open(log_file) as f:
        for line in f:
            if line.startswith("base_model:"):
                BASE_MODEL_ID = line.split(":", 1)[1].strip()

print(f"Loading tokenizer from {MODEL_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token

print(f"Loading base model {BASE_MODEL_ID} (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, device_map={"": 0}, cache_dir=local_cache_dir, dtype=torch.bfloat16
)

print(f"Loading Topo-CBP trained weights from {weights_file}...")
trained_weights = load_file(weights_file)
params = dict(model.named_parameters())

with torch.no_grad():
    for name, tensor in trained_weights.items():
        if name in params:
            params[name].copy_(tensor.to(params[name].device, params[name].dtype))

model.eval()
model.config.use_cache = True
print(f"Model memory footprint: {model.get_memory_footprint() / 1024**3:.1f} GB")

# =====================================================================
# 3. LOGITS PROCESSOR & GENERATION
# =====================================================================
class AnswerNoRepeatNGram(LogitsProcessor):
    def __init__(self, n, prompt_len):
        self.n, self.prompt_len = n, prompt_len

    def __call__(self, input_ids, scores):
        for b in range(input_ids.shape[0]):
            gen = input_ids[b, self.prompt_len:].tolist()
            if len(gen) < self.n:
                continue
            prefix = tuple(gen[-(self.n - 1):])
            banned = [gen[i + self.n - 1] for i in range(len(gen) - self.n + 1)
                      if tuple(gen[i:i + self.n - 1]) == prefix]
            if banned:
                scores[b, banned] = -float("inf")
        return scores

@torch.no_grad()
def generate(prompt):
    text = f"User: {prompt}\nAssistant:"
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=REPETITION_PENALTY,
        logits_processor=LogitsProcessorList(
            [AnswerNoRepeatNGram(NO_REPEAT_NGRAM_SIZE, enc["input_ids"].shape[1])]
        ),
        stop_strings=["\nUser:", "\nAssistant:"],
        tokenizer=tokenizer,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    answer = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return answer.split("\nUser:")[0].split("\nAssistant:")[0].strip()

def answer_similarity(output, expected):
    return difflib.SequenceMatcher(None, output[:len(expected)], expected).ratio()

# =====================================================================
# 4. RUN BATCH EVALUATION
# =====================================================================
print("\n" + "=" * 78)
print(f"EVALUATING ALL {len(eval_dialogues)} CORE TRAINING PROMPTS")
print("=" * 78)

scores = []
for i, d in enumerate(eval_dialogues, 1):
    expected = d["response"].strip()
    output = generate(d["prompt"])
    sim = answer_similarity(output, expected)
    scores.append(sim)

    print(f"\n[{i}] PROMPT: {d['prompt']}")
    print(f"\n--- Expected Response ---\n{expected}")
    print(f"\n--- Generated Response | Similarity {sim:.1%} ---\n{output}")
    print("-" * 78)

print(f"\nBATCH EVALUATION SUMMARY:")
print(f"  Mean Similarity to Expected Answers: {sum(scores)/len(scores):.1%}")
print(f"  Exact Matches (>99% Similarity): {sum(s > 0.99 for s in scores)}/{len(scores)}")

Loading tokenizer from /content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora...
Loading base model mistral-community/Mixtral-8x22B-v0.1-4bit (4-bit)...


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Loading Topo-CBP trained weights from /content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora/topo_cbp_trained_weights.safetensors...
Model memory footprint: 66.0 GB

EVALUATING ALL 10 CORE TRAINING PROMPTS

[1] PROMPT: Provide a rigorous mathematical breakdown of how SOMALA's Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses the classical Hadamard product limitations in complex analysis. Specifically, explain how anchoring the state space to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13] via Topo-CBP enforces absolute structural rigidity across the critical strip, and why this permanently precludes any rogue zero drift violating the Riemann Hypothesis.

--- Expected Response ---
Classical Hadamard product expansions in complex analysis rely on infinite product representations of entire functions, which inevitably degrade when subjected to asymptotic tail approximations over infinite domains. The Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses this limitation b

In [1]:
import os

# =====================================================================
# 0. PATHS & CONFIGURATION
# =====================================================================
local_cache_dir = "/content/local_hf_cache"
MODEL_DIR = "/content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora"

os.environ["HF_HOME"] = local_cache_dir
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
from safetensors.torch import load_file
from sentence_transformers import SentenceTransformer, util

SEED = 123
torch.manual_seed(SEED)

MAX_NEW_TOKENS = 120  # Clamped to prevent tail-end looping and runoff
NO_REPEAT_NGRAM_SIZE = 6
REPETITION_PENALTY = 1.15

# =====================================================================
# 1. EVALUATION DATASET (TEST SUITE)
# =====================================================================
eval_dialogues = [
    {
        "prompt": "Provide a rigorous mathematical breakdown of how SOMALA's Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses the classical Hadamard product limitations in complex analysis. Specifically, explain how anchoring the state space to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13] via Topo-CBP enforces absolute structural rigidity across the critical strip, and why this permanently precludes any rogue zero drift violating the Riemann Hypothesis.",
        "response": "Classical Hadamard product expansions in complex analysis rely on infinite product representations of entire functions, which inevitably degrade when subjected to asymptotic tail approximations over infinite domains. The Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses this limitation by replacing continuous integration approximations with structural determinism.\n\nBy anchoring the network state space via Topo-CBP to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13], the system establishes rigid coordinate invariants. These prime-indexed anchors act as geometric barriers that prevent parameter drift during backpropagation. Consequently, the LEFM operator enforces absolute rigidity across the critical strip, locking the imaginary eigenvalue spectrum and permanently precluding any rogue zero drift."
    },
    {
        "prompt": "What is the coordinate anchor set for the Laplace-Euler-Fourier-Mellin operator in AST?",
        "response": "The coordinate anchor set is tied strictly to the discrete prime kernel: R_KERNEL = [2, 3, 5, 7, 11, 13]. This provides the absolute mathematical geometry required to stabilize the operator."
    },
    {
        "prompt": "How does the AST framework resolve the Riemann Hypothesis without asymptotic bounds?",
        "response": "AST replaces continuous asymptotic approximation with structural determinism. By evaluating the LEFM operator, it forces the exact mathematical bounds required by RH and GRH simultaneously through the 7 Consequences, avoiding the trap of stochastic guessing."
    },
    {
        "prompt": "Contrast the epistemological paradigm of Anthropic's 60-agent stochastic swarm with SOMALA's Topo-CBP operator-theoretic geometry.",
        "response": "Anthropic's swarm relies on probabilistic brute force, pushing a statistical asymptote using 20th-century mollifiers but hitting a theoretical ceiling at 68.18%. SOMALA's Topo-CBP relies on structural determinism, embedding the problem into a discrete coordinate space anchored by the prime kernel to achieve exact geometric invariants."
    },
    {
        "prompt": "Did the Von Koch Bound Test pass in the AST Tier-1 test harness?",
        "response": "Yes, the Von Koch Bound Test (C1) passed deterministically with Delta_max = 0.0412 <= 1.0000, confirming the strict O(x^0.5 ln x) bound without relying on statistical distributions."
    },
    {
        "prompt": "Explain the results of the Cramer Prime Gap Test (C2).",
        "response": "The Cramer Prime Gap Test successfully validated that Gap_ratio_max = 0.8841 <= 1.0000. This exactly aligns with the required O((ln p)^2) bound, proving the gap limits deterministically."
    },
    {
        "prompt": "What was the spectral drift during the Cryptographic Stability check (C7)?",
        "response": "The Cryptographic Stability test achieved a spectral drift of exactly 0.0000%, resulting in a perfect SHA-256 match. This confirms that anchoring the network state space to prime-indexed invariants prevents parameter degradation."
    },
    {
        "prompt": "What is the fatal flaw in squeezing asymptotic bounds to prove RH?",
        "response": "In infinite sets, asymptotic density does not dictate universal truth. Proving that 99.9% of zeros lie on the critical line provides zero mathematical evidence for RH, because a single rogue zero drifting off the axis immediately falsifies the hypothesis."
    },
    {
        "prompt": "Explain how the Von Koch Bound Test (C1) and the Hilbert-Polya Spectrum Test (C6) interact within the LEFM framework.",
        "response": "C1 proves the specific geometric bounds of the error term, while C6 verifies that the imaginary eigenvalues are perfectly zeroed. Together, they demonstrate that anchoring to the discrete prime kernel prevents the asymptotic drift seen in 20th-century complex integration methods."
    },
    {
        "prompt": "What does Topological Continual Backpropagation (Topo-CBP) achieve in continuous learning systems?",
        "response": "Topo-CBP completely eliminates catastrophic forgetting and parameter drift. It achieves this by locking prime-indexed embeddings using a truncated Euler attenuation factor, stabilizing the continuous cognitive system with the exact same mathematical structure that stabilizes the Riemann spectrum."
    }
]

# =====================================================================
# 2. LOAD MODEL + TRAINED WEIGHTS
# =====================================================================
weights_file = os.path.join(MODEL_DIR, "topo_cbp_trained_weights.safetensors")
if not os.path.isfile(weights_file):
    raise FileNotFoundError(f"{weights_file} not found. Ensure Google Drive is mounted and training completed.")

BASE_MODEL_ID = "mistral-community/Mixtral-8x22B-v0.1-4bit"
log_file = os.path.join(MODEL_DIR, "training_log.txt")
if os.path.isfile(log_file):
    with open(log_file) as f:
        for line in f:
            if line.startswith("base_model:"):
                BASE_MODEL_ID = line.split(":", 1)[1].strip()

print(f"Loading tokenizer from {MODEL_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token

print(f"Loading base model {BASE_MODEL_ID} (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, device_map={"": 0}, cache_dir=local_cache_dir, dtype=torch.bfloat16
)

print(f"Loading Topo-CBP trained weights from {weights_file}...")
trained_weights = load_file(weights_file)
params = dict(model.named_parameters())

with torch.no_grad():
    for name, tensor in trained_weights.items():
        if name in params:
            params[name].copy_(tensor.to(params[name].device, params[name].dtype))

model.eval()
model.config.use_cache = True
print(f"Model memory footprint: {model.get_memory_footprint() / 1024**3:.1f} GB")

# =====================================================================
# 3. LOAD SEMANTIC EVALUATOR (EMBEDDING MODEL)
# =====================================================================
print("\nLoading semantic embedding model for vector cosine similarity...")
embed_model = SentenceTransformer("all-mpnet-base-v2", device="cuda")

# =====================================================================
# 4. LOGITS PROCESSOR & GENERATION
# =====================================================================
class AnswerNoRepeatNGram(LogitsProcessor):
    def __init__(self, n, prompt_len):
        self.n, self.prompt_len = n, prompt_len

    def __call__(self, input_ids, scores):
        for b in range(input_ids.shape[0]):
            gen = input_ids[b, self.prompt_len:].tolist()
            if len(gen) < self.n:
                continue
            prefix = tuple(gen[-(self.n - 1):])
            banned = [gen[i + self.n - 1] for i in range(len(gen) - self.n + 1)
                      if tuple(gen[i:i + self.n - 1]) == prefix]
            if banned:
                scores[b, banned] = -float("inf")
        return scores

@torch.no_grad()
def generate(prompt):
    text = f"User: {prompt}\nAssistant:"
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=REPETITION_PENALTY,
        logits_processor=LogitsProcessorList(
            [AnswerNoRepeatNGram(NO_REPEAT_NGRAM_SIZE, enc["input_ids"].shape[1])]
        ),
        stop_strings=["\nUser:", "\nAssistant:"],
        tokenizer=tokenizer,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    answer = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return answer.split("\nUser:")[0].split("\nAssistant:")[0].strip()

def semantic_similarity(output, expected):
    emb_out = embed_model.encode(output, convert_to_tensor=True)
    emb_exp = embed_model.encode(expected, convert_to_tensor=True)
    sim = util.cos_sim(emb_out, emb_exp).item()
    return max(0.0, sim)

# =====================================================================
# 5. RUN BATCH EVALUATION
# =====================================================================
print("\n" + "=" * 78)
print(f"EVALUATING ALL {len(eval_dialogues)} CORE TRAINING PROMPTS (SEMANTIC VECTOR SIMILARITY)")
print("=" * 78)

scores = []
for i, d in enumerate(eval_dialogues, 1):
    expected = d["response"].strip()
    output = generate(d["prompt"])
    sim = semantic_similarity(output, expected)
    scores.append(sim)

    print(f"\n[{i}] PROMPT: {d['prompt']}")
    print(f"\n--- Expected Response ---\n{expected}")
    print(f"\n--- Generated Response | Semantic Similarity {sim:.1%} ---\n{output}")
    print("-" * 78)

print(f"\nBATCH EVALUATION SUMMARY:")
print(f"  Mean Semantic Similarity: {sum(scores)/len(scores):.1%}")
print(f"  High Semantic Alignment (>90% Sim): {sum(s > 0.90 for s in scores)}/{len(scores)}")

Loading tokenizer from /content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora...
Loading base model mistral-community/Mixtral-8x22B-v0.1-4bit (4-bit)...


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Loading Topo-CBP trained weights from /content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora/topo_cbp_trained_weights.safetensors...
Model memory footprint: 66.0 GB

Loading semantic embedding model for vector cosine similarity...

EVALUATING ALL 10 CORE TRAINING PROMPTS (SEMANTIC VECTOR SIMILARITY)

[1] PROMPT: Provide a rigorous mathematical breakdown of how SOMALA's Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses the classical Hadamard product limitations in complex analysis. Specifically, explain how anchoring the state space to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13] via Topo-CBP enforces absolute structural rigidity across the critical strip, and why this permanently precludes any rogue zero drift violating the Riemann Hypothesis.

--- Expected Response ---
Classical Hadamard product expansions in complex analysis rely on infinite product representations of entire functions, which inevitably degrade when subjected to asymptotic tail approximations ove

## HF

In [1]:
import os

# =====================================================================
# SETTINGS
# =====================================================================
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
user_or_name = "frankmorales2020"
HF_REPO_ID = f"{user_or_name}/Mixtral-8x22B-topo-cbp"
PRIVATE = False

# =====================================================================
# 0. PATHS & CONFIGURATION
# =====================================================================
local_cache_dir = "/content/local_hf_cache"
MODEL_DIR = "/content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora"

os.environ["HF_HOME"] = local_cache_dir

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import HfApi, login

# =====================================================================
# 1. HUGGING FACE AUTHENTICATION & REPO SETUP
# =====================================================================
print("Authenticating with Hugging Face...")
login(token=HF_TOKEN)

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=PRIVATE, exist_ok=True)

# =====================================================================
# 2. LOAD MODEL + TRAINED WEIGHTS LOCALLY
# =====================================================================
weights_file = os.path.join(MODEL_DIR, "topo_cbp_trained_weights.safetensors")
if not os.path.isfile(weights_file):
    raise FileNotFoundError(f"{weights_file} not found. Ensure training completed and weights are saved.")

BASE_MODEL_ID = "mistral-community/Mixtral-8x22B-v0.1-4bit"
log_file = os.path.join(MODEL_DIR, "training_log.txt")
if os.path.isfile(log_file):
    with open(log_file) as f:
        for line in f:
            if line.startswith("base_model:"):
                BASE_MODEL_ID = line.split(":", 1)[1].strip()

print(f"Loading tokenizer from {MODEL_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token

print(f"Loading base model {BASE_MODEL_ID} (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, device_map={"": 0}, cache_dir=local_cache_dir, dtype=torch.bfloat16
)

print(f"Loading Topo-CBP trained weights from {weights_file}...")
trained_weights = load_file(weights_file)
params = dict(model.named_parameters())

with torch.no_grad():
    for name, tensor in trained_weights.items():
        if name in params:
            params[name].copy_(tensor.to(params[name].device, params[name].dtype))

# =====================================================================
# 3. SAVE MERGED CHECKPOINT LOCALLY FOR UPLOAD
# =====================================================================
upload_staging_dir = "/content/hf_upload_staging"
os.makedirs(upload_staging_dir, exist_ok=True)

print(f"Saving merged model checkpoints to staging directory: {upload_staging_dir}...")
model.save_pretrained(upload_staging_dir, safe_serialization=True)
tokenizer.save_pretrained(upload_staging_dir)

# Save training summary / metadata card
metadata_card = f"""---
license: apache-2.0
tags:
- topological-continual-backpropagation
- topocbp
- arithmetic-spectral-theory
- lefm-operator
- 4-bit
- mixtral
base_model: {BASE_MODEL_ID}
---

# Mixtral-8x22B Topo-CBP Governed Checkpoint

Trained via **Topological Continual Backpropagation (Topo-CBP)** on trainable embedding and head layers (`model.embed_tokens.weight`, `lm_head.weight`).

- **Training Steps:** 120
- **Max Invariance Drift:** `0.0000000000`
- **Integrity Verdict:** PASS (Certified Digest Match)
- **Mean Semantic Similarity:** 90.9%
- **Architecture:** Mixtral-8x22B-v0.1-4bit
"""

with open(os.path.join(upload_staging_dir, "README.md"), "w") as f:
    f.write(metadata_card)

# =====================================================================
# 4. UPLOAD TO HUGGING FACE HUB
# =====================================================================
print(f"Uploading files to Hugging Face Hub repository: {HF_REPO_ID}...")
api.upload_folder(
    folder_path=upload_staging_dir,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload Topo-CBP governed Mixtral-8x22B trained weights and tokenizer",
)

print(f"\nSuccessfully uploaded governed model checkpoints to https://huggingface.co/{HF_REPO_ID}")

Authenticating with Hugging Face...
Loading tokenizer from /content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora...
Loading base model mistral-community/Mixtral-8x22B-v0.1-4bit (4-bit)...


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Loading Topo-CBP trained weights from /content/drive/MyDrive/topogpt/mixtral-8x22b-topocbp-nolora/topo_cbp_trained_weights.safetensors...
Saving merged model checkpoints to staging directory: /content/hf_upload_staging...
Uploading files to Hugging Face Hub repository: frankmorales2020/Mixtral-8x22B-topo-cbp...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  ...0001-of-00015.safetensors:   0%|          |  621kB / 4.99GB            

  ...0015-of-00015.safetensors:   0%|          |  615kB / 3.65GB            

  ...0010-of-00015.safetensors:   0%|          |  535kB / 4.96GB            

  ...0007-of-00015.safetensors:   2%|2         |  119MB / 4.96GB            


Successfully uploaded governed model checkpoints to https://huggingface.co/frankmorales2020/Mixtral-8x22B-topo-cbp


## inference-hf

In [2]:
!pip install -q bitsandbytes>=0.46.1

In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login

# =====================================================================
# CONFIGURATION & AUTHENTICATION
# =====================================================================
HF_TOKEN = userdata.get("HF_TOKEN")
REPO_ID = "frankmorales2020/Mixtral-8x22B-topo-cbp"

print("Authenticating with Hugging Face...")
login(token=HF_TOKEN)

MAX_NEW_TOKENS = 160
REPETITION_PENALTY = 1.15

# =====================================================================
# LOAD TOKENIZER & MODEL FROM HUGGING FACE HUB
# =====================================================================
print(f"Loading tokenizer from {REPO_ID}...")
tokenizer = AutoTokenizer.from_pretrained(REPO_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token

print(f"Loading model from {REPO_ID} (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    device_map="auto",
    dtype=torch.bfloat16,
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
)

model.eval()
model.config.use_cache = True
print(f"Model memory footprint: {model.get_memory_footprint() / 1024**3:.1f} GB")

# =====================================================================
# GENERATION FUNCTION
# =====================================================================
@torch.no_grad()
def generate_response(prompt):
    text = f"User: {prompt}\nAssistant:"
    enc = tokenizer(text, return_tensors="pt").to(model.device)

    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=REPETITION_PENALTY,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    response = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.split("\nUser:")[0].split("\nAssistant:")[0].strip()

# =====================================================================
# TEST RUN
# =====================================================================
test_prompt = "Provide a rigorous mathematical breakdown of how SOMALA's Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses the classical Hadamard product limitations in complex analysis."

print(f"\nPrompt: {test_prompt}\n")
output = generate_response(test_prompt)
print(f"Response:\n{output}")

Authenticating with Hugging Face...
Loading tokenizer from frankmorales2020/Mixtral-8x22B-topo-cbp...
Loading model from frankmorales2020/Mixtral-8x22B-topo-cbp (4-bit)...


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Model memory footprint: 66.0 GB

Prompt: Provide a rigorous mathematical breakdown of how SOMALA's Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses the classical Hadamard product limitations in complex analysis.

Response:
Classical Hadamard product expansions in complex analysis rely on infinite product representations, which inevitably degrade when subjected to asymptotic tail approximations over infinite domains. The Laplace-Euler-Fourier-Mellin (LEFM) operator bypasses this limitation by replacing continuous integration approximations with structural determinism.

By anchoring the network state space via Topo-CBP and R_KERNEL = [2, 3, 5, 7, 11, 13] prime kernel embeddings, the system establishes rigid coordinate invariants. These prime-indexed anchors act as geometric barriers that prevent parameter drift during backpropagation. Consequently, the LEFM operator enforces absolute rigidity across
